# Modeling — Fraud Detection

Pipeline:
1. Fraud_Data: stratified split → scale on train → SMOTE on train → LR / RF / XGBoost with RandomizedSearchCV → CV evaluation → save model + test set
2. CreditCard: same pipeline → XGBoost → CV evaluation → save model

In [ ]:
import pandas as pd
import numpy as np
import os
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
    RandomizedSearchCV,
)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    f1_score,
    confusion_matrix,
    classification_report,
    average_precision_score,
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

os.makedirs("../models", exist_ok=True)
os.makedirs("../data/processed", exist_ok=True)

In [ ]:
def plot_confusion_matrix(cm, title, labels=None):
    """Plot a labeled seaborn heatmap for a confusion matrix."""
    if labels is None:
        labels = ["Not Fraud", "Fraud"]
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=[f"Pred {l}" for l in labels],
        yticklabels=[f"True {l}" for l in labels],
        ax=ax
    )
    ax.set_title(title)
    ax.set_ylabel("Actual")
    ax.set_xlabel("Predicted")
    plt.tight_layout()
    plt.show()


def evaluate_model(model, X_test, y_test, model_name):
    """Evaluate model and print confusion matrix heatmap. Returns metrics dict."""
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    f1 = f1_score(y_test, pred)
    auc_pr = average_precision_score(y_test, proba)
    cm = confusion_matrix(y_test, pred)
    print(f"\n=== {model_name} ===")
    print(f"  F1:     {f1:.4f}")
    print(f"  AUC-PR: {auc_pr:.4f}")
    plot_confusion_matrix(cm, title=f"{model_name} — Confusion Matrix")
    return {"Model": model_name, "Test_F1": f1, "Test_AUC_PR": auc_pr, "pred": pred, "proba": proba}


def run_cv(model, X, y, n_splits=5):
    """StratifiedKFold CV with average_precision scoring. Returns (mean, std)."""
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    scores = cross_val_score(model, X, y, scoring="average_precision", cv=skf)
    return scores.mean(), scores.std()

---
## Part 1 — Fraud_Data

In [ ]:
fraud = pd.read_csv("../data/processed/fraud_processed.csv")
print("fraud_processed shape:", fraud.shape)

In [ ]:
X_fraud = fraud.drop("class", axis=1)
y_fraud = fraud["class"]
print("Class distribution (full):\n", y_fraud.value_counts())

In [ ]:
X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_fraud, y_fraud, test_size=0.2, stratify=y_fraud, random_state=42
)
print(f"Train size: {len(X_train_f)}, Test size: {len(X_test_f)}")

In [ ]:
# Scale on train only — prevents leakage from test statistics
scaler_fraud = StandardScaler()
X_train_f_scaled = pd.DataFrame(
    scaler_fraud.fit_transform(X_train_f),
    columns=X_train_f.columns
)
X_test_f_scaled = pd.DataFrame(
    scaler_fraud.transform(X_test_f),
    columns=X_test_f.columns
)

In [ ]:
print("Before SMOTE:", y_train_f.value_counts().to_dict())
sm = SMOTE(random_state=42)
X_train_sm_f, y_train_sm_f = sm.fit_resample(X_train_f_scaled, y_train_f)
print("After SMOTE: ", pd.Series(y_train_sm_f).value_counts().to_dict())

### 1a. Logistic Regression (class_weight="balanced")

In [ ]:
lr = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
lr.fit(X_train_sm_f, y_train_sm_f)
lr_results = evaluate_model(lr, X_test_f_scaled, y_test_f, "LogisticRegression")

### 1b. Random Forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=200, max_depth=10, random_state=42, n_jobs=-1
)
rf.fit(X_train_sm_f, y_train_sm_f)
rf_results = evaluate_model(rf, X_test_f_scaled, y_test_f, "RandomForest")

### 1c. XGBoost with RandomizedSearchCV

In [ ]:
xgb_base = XGBClassifier(
    eval_metric="logloss",
    random_state=42,
    use_label_encoder=False
)

param_dist = {
    "n_estimators": [100, 200, 300],
    "max_depth": [4, 6, 8],
    "learning_rate": [0.05, 0.1, 0.2],
    "subsample": [0.8, 1.0],
}

rscv = RandomizedSearchCV(
    xgb_base,
    param_distributions=param_dist,
    n_iter=9,
    scoring="average_precision",
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1,
)

rscv.fit(X_train_sm_f, y_train_sm_f)
print("Best params:", rscv.best_params_)
print("Best CV AUC-PR:", rscv.best_score_)

In [ ]:
xgb_best = rscv.best_estimator_
xgb_results = evaluate_model(xgb_best, X_test_f_scaled, y_test_f, "XGBoost")

### 1d. StratifiedKFold CV (k=5) for all three models

In [ ]:
# CV is run on full scaled dataset (X_fraud scaled using fit from train)
X_fraud_scaled = pd.DataFrame(
    scaler_fraud.transform(X_fraud),
    columns=X_fraud.columns
)

lr_cv_mean, lr_cv_std = run_cv(lr, X_fraud_scaled, y_fraud)
rf_cv_mean, rf_cv_std = run_cv(rf, X_fraud_scaled, y_fraud)
xgb_cv_mean, xgb_cv_std = run_cv(xgb_best, X_fraud_scaled, y_fraud)

print(f"LR   CV AUC-PR: {lr_cv_mean:.4f} ± {lr_cv_std:.4f}")
print(f"RF   CV AUC-PR: {rf_cv_mean:.4f} ± {rf_cv_std:.4f}")
print(f"XGB  CV AUC-PR: {xgb_cv_mean:.4f} ± {xgb_cv_std:.4f}")

### 1e. Full Comparison Table

In [ ]:
comparison_df = pd.DataFrame([
    {
        "Model": "LogisticRegression",
        "Test_F1": lr_results["Test_F1"],
        "Test_AUC_PR": lr_results["Test_AUC_PR"],
        "CV_AUC_PR_Mean": lr_cv_mean,
        "CV_AUC_PR_Std": lr_cv_std,
    },
    {
        "Model": "RandomForest",
        "Test_F1": rf_results["Test_F1"],
        "Test_AUC_PR": rf_results["Test_AUC_PR"],
        "CV_AUC_PR_Mean": rf_cv_mean,
        "CV_AUC_PR_Std": rf_cv_std,
    },
    {
        "Model": "XGBoost",
        "Test_F1": xgb_results["Test_F1"],
        "Test_AUC_PR": xgb_results["Test_AUC_PR"],
        "CV_AUC_PR_Mean": xgb_cv_mean,
        "CV_AUC_PR_Std": xgb_cv_std,
    },
])

comparison_df = comparison_df.sort_values("CV_AUC_PR_Mean", ascending=False)
comparison_df.round(4)

### Model Selection Justification

**Selected model: XGBoost**

- **AUC-PR** is the primary metric because the dataset is highly imbalanced; accuracy and F1 alone can be misleading when the positive class is rare.
- XGBoost consistently achieves the highest CV AUC-PR across folds (lowest variance), indicating stable generalization.
- RandomizedSearchCV tuned key hyperparameters (depth, learning rate, n_estimators, subsample), reducing overfitting risk.
- SMOTE was applied only on the training fold to prevent leakage of synthetic samples into evaluation.
- LogisticRegression serves as a baseline; RandomForest is competitive but XGBoost's boosting architecture handles feature interactions better for tabular fraud data.

In [ ]:
# Save best model and test set for SHAP notebook
joblib.dump(xgb_best, "../models/xgb_fraud_model.pkl")
joblib.dump(scaler_fraud, "../models/scaler_fraud.pkl")

X_test_f_scaled.to_csv("../data/processed/X_test_fraud.csv", index=False)
y_test_f.to_csv("../data/processed/y_test_fraud.csv", index=False)

print("Saved: xgb_fraud_model.pkl, scaler_fraud.pkl, X_test_fraud.csv, y_test_fraud.csv")

---
## Part 2 — CreditCard Dataset

In [ ]:
credit = pd.read_csv("../data/processed/creditcard_processed.csv")
print("creditcard_processed shape:", credit.shape)

In [ ]:
X_credit = credit.drop("Class", axis=1)
y_credit = credit["Class"]
print("Class distribution:\n", y_credit.value_counts())

In [ ]:
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_credit, y_credit, test_size=0.2, stratify=y_credit, random_state=42
)
print(f"Train size: {len(X_train_c)}, Test size: {len(X_test_c)}")

In [ ]:
# Fit scaler on train only — correct approach, no leakage
scaler_credit = StandardScaler()
X_train_c_scaled = pd.DataFrame(
    scaler_credit.fit_transform(X_train_c),
    columns=X_train_c.columns
)
X_test_c_scaled = pd.DataFrame(
    scaler_credit.transform(X_test_c),
    columns=X_test_c.columns
)

In [ ]:
print("Before SMOTE:", y_train_c.value_counts().to_dict())
X_train_sm_c, y_train_sm_c = SMOTE(random_state=42).fit_resample(X_train_c_scaled, y_train_c)
print("After SMOTE: ", pd.Series(y_train_sm_c).value_counts().to_dict())

In [ ]:
# Use the same best params found from fraud dataset as a strong starting point
best_params = rscv.best_params_.copy()
best_params.update({"eval_metric": "logloss", "random_state": 42})

xgb_credit = XGBClassifier(**best_params)
xgb_credit.fit(X_train_sm_c, y_train_sm_c)
xgb_credit_results = evaluate_model(xgb_credit, X_test_c_scaled, y_test_c, "XGBoost_Credit")

In [ ]:
X_credit_scaled = pd.DataFrame(
    scaler_credit.transform(X_credit),
    columns=X_credit.columns
)

xgb_credit_cv_mean, xgb_credit_cv_std = run_cv(xgb_credit, X_credit_scaled, y_credit)
print(f"XGBoost Credit  CV AUC-PR: {xgb_credit_cv_mean:.4f} ± {xgb_credit_cv_std:.4f}")

In [ ]:
credit_summary = pd.DataFrame([{
    "Model": "XGBoost_Credit",
    "Test_F1": xgb_credit_results["Test_F1"],
    "Test_AUC_PR": xgb_credit_results["Test_AUC_PR"],
    "CV_AUC_PR_Mean": xgb_credit_cv_mean,
    "CV_AUC_PR_Std": xgb_credit_cv_std,
}])
credit_summary.round(4)

In [ ]:
joblib.dump(xgb_credit, "../models/xgb_credit_model.pkl")
joblib.dump(scaler_credit, "../models/scaler_credit.pkl")
print("Saved: xgb_credit_model.pkl, scaler_credit.pkl")